# PaperMind writer: QLoRA SFT + DPO on Mistral 7B

Run on an A100 (Colab Pro or RunPod). About 2 hours of GPU time.

Before this: on your laptop, run `training.generate`, `training.filter` and `training.build` (see `training/README.md`), then commit or upload `data/training/*.jsonl`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/an301/papermind.git
%cd papermind
!pip install -q -r training/requirements-train.txt

Upload the four dataset files if they aren't in the repo: `data/training/{sft,dpo}_{train,eval}.jsonl`.

In [ ]:
!ls -la data/training/*.jsonl && wc -l data/training/*.jsonl

In [ ]:
from huggingface_hub import login

import wandb

login()  # a write token, to push the adapter
wandb.login()

## 1. SFT: learn what a good answer at each level looks like

Watch W&B: train loss should fall from ~1.4–1.8 to ~0.9–1.2. If eval loss rises while train loss falls, keep the earlier checkpoint (the best one by eval loss is loaded at the end).

In [ ]:
!python -m training.train --stage sft --out out/sft

## 2. DPO: prefer the right level over a fluent answer at the wrong one

The loss must start at **0.693** (ln 2). `rewards/accuracies` should climb above 0.5 toward 0.8–0.9 and `rewards/margins` grow to ~1–3.

In [ ]:
!python -m training.train --stage dpo --sft-adapter out/sft --out out/dpo --push-to-hub an301/papermind-writer

## 3. Serve it

Colab disconnects wipe the disk, so the adapter was pushed to the Hub above. On any GPU box:

```bash
pip install vllm
vllm serve mistralai/Mistral-7B-Instruct-v0.3 --enable-lora \
  --lora-modules papermind=an301/papermind-writer --max-lora-rank 16 --port 8001
```

Then in PaperMind's `.env`: `WRITER_BACKEND=mistral`, `MISTRAL_BASE_URL=http://<gpu-box>:8001/v1`.